In [ ]:
import tarfile

with tarfile.open("/content/1774118365245-cv-corpus-25.0-2026-03-09-az.tar.gz", "r:gz") as tar:
    tar.extractall("common_voice_az")

/tmp/ipykernel_4728/2350319817.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("common_voice_az")


In [94]:
import os
import pandas as pd
from datasets import Dataset, Audio

DATA_DIR = "/content/common_voice_az/cv-corpus-25.0-2026-03-09/az/"
CLIPS_DIR = f"{DATA_DIR}/clips"

df = pd.read_csv(f"{DATA_DIR}/train.tsv", sep='\t')


df['audio'] = df['path'].apply(lambda p: os.path.join(CLIPS_DIR, p))

df = df[['audio', 'sentence']]

dataset = Dataset.from_pandas(df)

dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

dataset[0]

{'audio': <datasets.features._torchcodec.AudioDecoder at 0x7dbbf1114680>,
 'sentence': 'Məqbərənin yaxınlığında ölməz şairin yaratdığı əsərləri tərənnüm edən tuncdan abidə yerləşir.'}

In [95]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = "openai/whisper-small"

if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

model.eval()

print("Device: ", device)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Device:  cuda


In [96]:
forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="az",
    task='transcribe'
)

In [ ]:
predictions = []
references = []

for i, example in enumerate(dataset):
    audio = example['audio']

    input_features = processor(
        audio['array'],
        sampling_rate=audio['sampling_rate'],
        return_tensors='pt'
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids
            )

    pred_text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    predictions.append(pred_text)
    references.append(example['sentence'])


    print(f"{i + 1}/{len(dataset)}")
    print(f'Prediction: {pred_text}')
    print(f'Reference: {example["sentence"]}')

1/215
Prediction:  Məqibəranın yaxınlığında ölmək şəhərin yaratıq əsərlər, tərəndi medin Tuncda nabidə yerləşir.
Reference: Məqbərənin yaxınlığında ölməz şairin yaratdığı əsərləri tərənnüm edən tuncdan abidə yerləşir.
2/215
Prediction:  Hava da krissallaşma suyunu tədəcəndirir.
Reference: Havada kristallaşma suyunu tədricən itirir.
3/215
Prediction:  Bir çox şək televizya proqramlarında apartı kimi çox eləyik.
Reference: Bir çox uşaq televiziya proqramlarında aparıcı kimi çıxış edib.
4/215
Prediction:  Xəzri zamanı başı və Sungar şəhirlənin havası zərələri rəz və tutun təminərinir.
Reference: Xəzri zamanı Bakı və Sumqayıt şəhərlərinin havası zərərli qaz və tüstüdən təmizlənir.
5/215
Prediction:  Bölçif 5 yolunun bir qolla oradan getirdi.
Reference: Böyük İpək yolunun bir qolu oradan keçirdi.
6/215
Prediction:  Etika təqqında.
Reference: Etiqad haqqında.
7/215
Prediction:  Bu dildə yazılan program az yerdir və tez yerini yetişirilər.
Reference: Bu dildə yazılan proqram az yer tutur və t

In [ ]:
import re

def normalize_text(text):
    text = text.lower().strip()

    # keep Azerbaijani letters, numbers, and spaces
    text = re.sub(r"[^\w\səıöüçğş]", " ", text)

    # normalize multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


norm_predictions = [normalize_text(x) for x in predictions]
norm_references = [normalize_text(x) for x in references]

In [ ]:
!pip install -q evaluate jiwer

In [ ]:
import evaluate

wer_metric = evaluate.load('wer')
cer_metric = evaluate.load('cer')

wer = wer_metric.compute(predictions=norm_predictions, references=norm_references)
cer = cer_metric.compute(predictions=norm_predictions, references=norm_references)

print(f'WER: {wer}')
print(f'CER: {cer}')

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from dataclasses import dataclass
import evaluate
import pandas as pd
import matplotlib.pyplot as plt

small_dataset = dataset.shuffle(seed=42).select(range(200))

split = small_dataset.train_test_split(test_size=0.2, seed=42)

train_raw = split["train"]
valid_raw = split["test"]

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_features"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    ).input_features[0]

    batch["labels"] = processor.tokenizer(
        batch["sentence"]
    ).input_ids

    return batch

In [ ]:
train_ds = train_raw.map(prepare_dataset, remove_columns=train_raw.column_names)
valid_ds = valid_raw.map(prepare_dataset, remove_columns=valid_raw.column_names)

In [ ]:
@dataclass
class DataCollator:
    processor: object

    def __call__(self, features):
        audio_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(audio_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

In [ ]:
data_collator = DataCollator(processor)

In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    preds = processor.batch_decode(pred_ids, skip_special_tokens=True)
    refs = processor.batch_decode(label_ids, skip_special_tokens=True)

    preds = [normalize_text(x) for x in preds]
    refs = [normalize_text(x) for x in refs]

    return {
        "wer": wer_metric.compute(predictions=preds, references=refs),
        "cer": cer_metric.compute(predictions=preds, references=refs),
    }

In [ ]:
base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
base_model.eval()
ft_model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
ft_model.eval()

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper_az_ft",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=1e-5,

    predict_with_generate=True,
    generation_max_length=128,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    logging_steps=5,
    save_total_limit=1,
    report_to="none"
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./best_whisper_az")
processor.save_pretrained("./best_whisper_az")

In [ ]:
logs = pd.DataFrame(trainer.state.log_history)

In [ ]:
loss_logs = logs[logs["loss"].notna()]

plt.plot(loss_logs["step"], loss_logs["loss"])
plt.title("Training Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.show()

In [ ]:
eval_logs = logs[logs["eval_wer"].notna()]

plt.plot(eval_logs["epoch"], eval_logs["eval_wer"], label="WER")
plt.plot(eval_logs["epoch"], eval_logs["eval_cer"], label="CER")
plt.title("Validation WER/CER")
plt.xlabel("Epoch")
plt.legend()
plt.show()

In [ ]:
fine_tuned_model = WhisperForConditionalGeneration.from_pretrained("./best_whisper_az").to(device)
fine_tuned_model.eval()

In [88]:
test_samples = valid_raw

In [89]:
def predict(model, data):
    preds = []
    refs = []

    for example in data:
        audio = example["audio"]

        input_features = processor(
            audio["array"],
            sampling_rate=audio["sampling_rate"],
            return_tensors="pt"
        ).input_features.to(device)

        with torch.no_grad():
            ids = model.generate(
                input_features,
                forced_decoder_ids=forced_decoder_ids
            )

        text = processor.batch_decode(ids, skip_special_tokens=True)[0]

        preds.append(text)
        refs.append(example["sentence"])

    return preds, refs

In [90]:
base_preds, refs = predict(model, test_samples)
ft_preds, _ = predict(fine_tuned_model, test_samples)

In [91]:
refs_norm = [normalize_text(x) for x in refs]
base_norm = [normalize_text(x) for x in base_preds]
ft_norm = [normalize_text(x) for x in ft_preds]

In [92]:
comparison = pd.DataFrame([
    {
        "model": "Base Whisper",
        "WER": wer_metric.compute(predictions=base_norm, references=refs_norm),
        "CER": cer_metric.compute(predictions=base_norm, references=refs_norm),
    },
    {
        "model": "Fine-tuned Whisper",
        "WER": wer_metric.compute(predictions=ft_norm, references=refs_norm),
        "CER": cer_metric.compute(predictions=ft_norm, references=refs_norm),
    }
])

comparison

,model,WER,CER
0,Base Whisper,0.50495,0.140318
1,Fine-tuned Whisper,0.39934,0.093545


In [93]:
pd.DataFrame({
    "reference": refs,
    "base_prediction": base_preds,
    "fine_tuned_prediction": ft_preds
})

,reference,base_prediction,fine_tuned_prediction
0,Böyük İpək yolunun bir qolu oradan keçirdi.,Bölçif 5 yolunun bir qolla oradan getirdi.,Böyük ihbəş yolunun bir qolu oradan getirdi.
1,Odun içində oynayır.,Odun Çin'də oynayır.,Odun Çindav enayr.
2,Dörd kitabın birinə də yiyə durmadın.,4 tabın birini də ye durmadın.,Dört kitabın birinə də yiyə durmadın.
3,Təşkilatın hədəfi imperatorun mütləq hakimiyyə...,Təşkilatın hədəfə imperatorun mükləq hakimiyy...,Təşkilatın hədəfə imperatorun mükləq hakimiyyə...
4,El Arslanın hakimiyyəti Böyük Səlcuq Dövlətini...,El Arslanın hakimiyyəti Büyüş Səzültlük dövlə...,El Arslanın hakimiyyəti Büyüş Səvcud dövlətini...
5,Keçmiş adı Qəhrəmanlı olmuşdur.,70 adı qəhrəmanlı olmuştur.,Ketmiş adı qəhramanlı olmuşdur.
6,Böyük ədədlər arasında sadə ədədi tapmaq üçün ...,Büyüc ədədilər arasında sade edədi tarqmalı ü...,Böyüş ədədilər arasında sadədədə tapmaq için b...
7,Domen iki il müddətinə pulsuz verilir.,Domain içi il müdətlə pulsuz verilir.,Domain iki il müddətnə pulsuz verilir.
8,Ondan bir oğulu olar.,Ondan bir oğulub olar.,Ondan bir oğlu olar.
9,Parkın ərazisində sahəsi üç hektar olan süni g...,Parkın ərazisində sahəsi üç hektar olan sünni...,Parkın ərazisində sahəsi üç hektar olan sünü g...
